In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gymnasium as gym

# Prevent excessive CPU threads
torch.set_num_threads(1)

# Parameters
H, BATCH, GAMMA, TAU, LR = 128, 128, 0.99, 0.005, 3e-4


# ---------------- ACTOR ----------------
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim, max_a):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(s_dim, H),
            nn.ReLU(),
            nn.Linear(H, H),
            nn.ReLU()
        )

        self.mean = nn.Linear(H, a_dim)
        self.log_std = nn.Linear(H, a_dim)
        self.max_a = max_a

    def sample(self, s):
        x = self.net(s)

        mean = self.mean(x)
        log_std = self.log_std(x).clamp(-20, 2)

        dist = torch.distributions.Normal(
            mean, log_std.exp()
        )

        z = dist.rsample()
        tanh_z = torch.tanh(z)

        action = tanh_z * self.max_a

        log_prob = (
            dist.log_prob(z)
            - torch.log(1 - tanh_z.pow(2) + 1e-6)
        ).sum(-1, keepdim=True)

        return action, log_prob


# ---------------- CRITIC ----------------
class Critic(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(s_dim + a_dim, H),
            nn.ReLU(),
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Linear(H, 1)
        )

    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=-1))


# ---------------- SAC ----------------
class SAC:
    def __init__(self, s_dim, a_dim, max_a):

        self.actor = Actor(s_dim, a_dim, max_a)

        self.q1 = Critic(s_dim, a_dim)
        self.q2 = Critic(s_dim, a_dim)

        self.tq1 = Critic(s_dim, a_dim)
        self.tq2 = Critic(s_dim, a_dim)

        self.tq1.load_state_dict(self.q1.state_dict())
        self.tq2.load_state_dict(self.q2.state_dict())

        self.opt_a = optim.Adam(
            self.actor.parameters(), lr=LR
        )

        self.opt_q1 = optim.Adam(
            self.q1.parameters(), lr=LR
        )

        self.opt_q2 = optim.Adam(
            self.q2.parameters(), lr=LR
        )

        self.log_alpha = torch.zeros(
            1, requires_grad=True
        )

        self.opt_alpha = optim.Adam(
            [self.log_alpha], lr=LR
        )

        self.target_entropy = -a_dim

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def act(self, s):
        with torch.no_grad():
            a, _ = self.actor.sample(
                torch.FloatTensor(s).unsqueeze(0)
            )

        return a.squeeze(0).numpy()

    def update(self, buf):

        if len(buf) < BATCH:
            return

        s, a, r, s2, d = buf.sample(BATCH)

        # Target Q
        with torch.no_grad():

            a2, logp2 = self.actor.sample(s2)

            q_target = torch.min(
                self.tq1(s2, a2),
                self.tq2(s2, a2)
            ) - self.alpha.detach() * logp2

            y = r + GAMMA * (1 - d) * q_target

        # Critic 1
        loss1 = ((self.q1(s, a) - y) ** 2).mean()

        self.opt_q1.zero_grad()
        loss1.backward()
        self.opt_q1.step()

        # Critic 2
        loss2 = ((self.q2(s, a) - y) ** 2).mean()

        self.opt_q2.zero_grad()
        loss2.backward()
        self.opt_q2.step()

        # Actor
        new_a, logp = self.actor.sample(s)

        q_min = torch.min(
            self.q1(s, new_a),
            self.q2(s, new_a)
        )

        actor_loss = (
            self.alpha.detach() * logp - q_min
        ).mean()

        self.opt_a.zero_grad()
        actor_loss.backward()
        self.opt_a.step()

        # Alpha
        alpha_loss = -(
            self.log_alpha *
            (logp + self.target_entropy).detach()
        ).mean()

        self.opt_alpha.zero_grad()
        alpha_loss.backward()
        self.opt_alpha.step()

        # Soft update
        for q, tq in [
            (self.q1, self.tq1),
            (self.q2, self.tq2)
        ]:

            for p, tp in zip(
                q.parameters(),
                tq.parameters()
            ):
                tp.data.mul_(1 - TAU)
                tp.data.add_(TAU * p.data)


# ---------------- REPLAY BUFFER ----------------
class Buffer:
    def __init__(self, cap=100000):
        self.d = deque(maxlen=cap)

    def push(self, *x):
        self.d.append(x)

    def sample(self, n):

        batch = random.sample(self.d, n)

        s, a, r, s2, d = zip(*batch)

        return (
            torch.FloatTensor(np.array(s)),
            torch.FloatTensor(np.array(a)),
            torch.FloatTensor(np.array(r)).unsqueeze(1),
            torch.FloatTensor(np.array(s2)),
            torch.FloatTensor(np.array(d)).unsqueeze(1)
        )

    def __len__(self):
        return len(self.d)


# ---------------- ENVIRONMENT ----------------
env = gym.make("Pendulum-v1")

agent = SAC(
    env.observation_space.shape[0],
    env.action_space.shape[0],
    float(env.action_space.high[0])
)

buf = Buffer()

reward_history = []


# ---------------- TRAINING ----------------
for ep in range(50):

    s, _ = env.reset()
    done = False
    ep_r = 0

    while not done:

        if len(buf) < 500:
            a = env.action_space.sample()
        else:
            a = agent.act(s)

        s2, r, term, trunc, _ = env.step(a)

        done = term or trunc

        buf.push(
            s, a, r, s2, float(done)
        )

        s = s2
        ep_r += r

        agent.update(buf)

    reward_history.append(ep_r)

    print(
        f"Episode {ep + 1}: "
        f"reward={ep_r:.1f}, "
        f"alpha={agent.alpha.item():.3f}"
    )


# ---------------- FINAL RESULT ----------------
last_n = np.array(reward_history[-10:])

mean_reward = np.mean(last_n)

sem_reward = (
    np.std(last_n, ddof=1) /
    np.sqrt(len(last_n))
)

print("\nFinal performance:")
print(
    f"(last {len(last_n)} episodes): "
    f"{mean_reward:.1f} +/- {sem_reward:.1f} "
    f"(mean +/- SEM)"
)

env.close()

Episode 1: reward=-1067.7, alpha=0.978
Episode 2: reward=-1399.4, alpha=0.922
Episode 3: reward=-921.8, alpha=0.870
Episode 4: reward=-1300.3, alpha=0.820
Episode 5: reward=-1518.2, alpha=0.775
Episode 6: reward=-1253.3, alpha=0.733
Episode 7: reward=-1794.3, alpha=0.694
Episode 8: reward=-1718.7, alpha=0.658
Episode 9: reward=-1562.0, alpha=0.624
Episode 10: reward=-1772.4, alpha=0.591
Episode 11: reward=-1465.7, alpha=0.559
Episode 12: reward=-1433.1, alpha=0.529
Episode 13: reward=-1698.2, alpha=0.502
Episode 14: reward=-968.5, alpha=0.475
Episode 15: reward=-1022.4, alpha=0.451
Episode 16: reward=-1154.2, alpha=0.428
Episode 17: reward=-1110.6, alpha=0.407
Episode 18: reward=-1075.9, alpha=0.388
Episode 19: reward=-1092.5, alpha=0.370
Episode 20: reward=-1189.6, alpha=0.352
Episode 21: reward=-1269.1, alpha=0.336
Episode 22: reward=-1399.7, alpha=0.321
Episode 23: reward=-1196.8, alpha=0.306
Episode 24: reward=-793.2, alpha=0.293
Episode 25: reward=-1108.4, alpha=0.281
Episode 26: 